# 5. Ground truth: the contributing tissue

For every recording at the ground-truth placements, each tissue element's signed share of the signal
energy, from the same openCARP lead fields as the electrograms (FM-14, FM-23). Shares sum to 1; nothing is cut off; negative
shares (cancellation) are kept. Reported: near/far (4 mm convention, FM-30) x endocardial/epicardial classes (FM-15), the
transmural profile per 0.1 mm layer, lateral reach, shape descriptors, and the full 3D fields
(`groundTruth/fields_<condition>.npz`, `groundTruth/shares_<condition>_gap<g>mm.vtk` for ParaView).

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v4_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

In [ ]:
checks = run.runStage("groundTruth")
checks

In [ ]:
summary = pd.read_csv(run.out / "groundTruth" / "summary.csv")
classes = [c for c in summary.columns if c.startswith("mass_")]
summary.groupby(["condition", "kind", "gapMm"])[classes].median()

In [ ]:
from efov.groundtruth_figures import makeGroundTruthFigures
for path in makeGroundTruthFigures(run, run.folder("figures")):
    display(Image(str(path)))